# Lab 13-2 Using MLFlow to track experiments

This notebook provides a quick overview of machine learning model training on Databricks. To train models, you can use libraries like scikit-learn that are preinstalled on the Databricks Runtime for Machine Learning. In addition, you can use MLflow to track the trained models.

This tutorial covers training a simple classification model with MLflow tracking

### Requirements

For a more detaile example that includes model lifecycle management and model inference, see the ML End to End Example ([AWS](https://docs.databricks.com/applications/mlflow/end-to-end-example.html)|[Azure](https://docs.microsoft.com/azure/databricks/applications/mlflow/end-to-end-example)|[GCP](https://docs.gcp.databricks.com/applications/mlflow/end-to-end-example.html)).

## Step 1: Preparation

### Import Libraries
Import the necessary libraries. These libraries are preinstalled on Databricks Runtime for Machine Learning ([AWS](https://docs.databricks.com/runtime/mlruntime.html)|[Azure](https://docs.microsoft.com/azure/databricks/runtime/mlruntime)|[GCP](https://docs.gcp.databricks.com/runtime/mlruntime.html)) clusters and are tuned for compatibility and performance.

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.ensemble

### Load data
The tutorial uses a dataset describing different wine samples. The [dataset](https://archive.ics.uci.edu/ml/datasets/Wine) is from the UCI Machine Learning Repository and is included in DBFS ([AWS](https://docs.databricks.com/data/databricks-file-system.html)|[Azure](https://docs.microsoft.com/azure/databricks/data/databricks-file-system)|[GCP](https://docs.gcp.databricks.com/data/databricks-file-system.html)).
The goal is to classify red and white wines by their quality. 

- `/databricks-datasets/wine-quality/winequality-white.csv`
- `/databricks-datasets/wine-quality/winequality-red.csv`


### View the format of the input data

### 1. View the format of the input data (using !head)

In [ ]:
white_quality = "/databricks-datasets/wine-quality/winequality-white.csv"
red_quality = "/databricks-datasets/wine-quality/winequality-red.csv"

In [ ]:
!head $white_quality

"fixed acidity";"volatile acidity";"citric acid";"residual sugar";"chlorides";"free sulfur dioxide";"total sulfur dioxide";"density";"pH";"sulphates";"alcohol";"quality"
7;0.27;0.36;20.7;0.045;45;170;1.001;3;0.45;8.8;6
6.3;0.3;0.34;1.6;0.049;14;132;0.994;3.3;0.49;9.5;6
8.1;0.28;0.4;6.9;0.05;30;97;0.9951;3.26;0.44;10.1;6
7.2;0.23;0.32;8.5;0.058;47;186;0.9956;3.19;0.4;9.9;6
7.2;0.23;0.32;8.5;0.058;47;186;0.9956;3.19;0.4;9.9;6
8.1;0.28;0.4;6.9;0.05;30;97;0.9951;3.26;0.44;10.1;6
6.2;0.32;0.16;7;0.045;30;136;0.9949;3.18;0.47;9.6;6
7;0.27;0.36;20.7;0.045;45;170;1.001;3;0.45;8.8;6
6.3;0.3;0.34;1.6;0.049;14;132;0.994;3.3;0.49;9.5;6


In [ ]:
!head $red_quality

"fixed acidity";"volatile acidity";"citric acid";"residual sugar";"chlorides";"free sulfur dioxide";"total sulfur dioxide";"density";"pH";"sulphates";"alcohol";"quality"
7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5
7.8;0.88;0;2.6;0.098;25;67;0.9968;3.2;0.68;9.8;5
7.8;0.76;0.04;2.3;0.092;15;54;0.997;3.26;0.65;9.8;5
11.2;0.28;0.56;1.9;0.075;17;60;0.998;3.16;0.58;9.8;6
7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5
7.4;0.66;0;1.8;0.075;13;40;0.9978;3.51;0.56;9.4;5
7.9;0.6;0.06;1.6;0.069;15;59;0.9964;3.3;0.46;9.4;5
7.3;0.65;0;1.2;0.065;15;21;0.9946;3.39;0.47;10;7
7.8;0.58;0.02;2;0.073;9;18;0.9968;3.36;0.57;9.5;7


In [ ]:
# Load and preprocess data
white_wine = pd.read_csv(white_quality, sep=';')
red_wine = pd.read_csv(red_quality, sep=';')

# merge the two datasets into one file, with a dummy column
white_wine['is_red'] = 0.0
red_wine['is_red'] = 1.0
data_df = pd.concat([white_wine, red_wine], axis=0)

# Define classification labels based on the wine quality
data_labels = (data_df['quality'] >= 7)
data_df = data_df.drop(['quality'], axis=1)

# Split 80/20 train-test
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
  data_df,
  data_labels,
  test_size=0.2,
  random_state=1
)

### 2. View first rows of the `X_train`, `X_test`

In [ ]:
X_train.head()

In [ ]:
X_test.head()

## Step 2: Train a classification model

[MLflow tracking](https://www.mlflow.org/docs/latest/tracking.html) allows you to organize your machine learning training code, parameters, and models. 

You can enable automatic MLflow tracking by using [*autologging*](https://www.mlflow.org/docs/latest/tracking.html#automatic-logging).
```python
# Enable MLflow autologging for this notebook
mlflow.autolog()
```

The generic autolog function `mlflow.autolog()` enables autologging for each supported library you have installed as soon as you import it. Supported libraries include:

- Keras
- LightGBM
- PyTorch
- Scikit-learn
- Spark
- XGBoost, and so on

It typically log the following information:
- **Metrics** - MLflow pre-selects a set of metrics to log, based on what model and library you use
- **Parameters** - hyper params specified for the training, plus default values provided by the library if not explicitly set
- **Model Signature** - logs Model signature instance, which describes input and output schema of the model
- **Artifacts** - e.g. model checkpoints
- **Dataset** - dataset object used for training (if applicable), such as tensorflow.data.Dataset



### 3. Enable MLflow autologging for sklearn within this notebook using

```python
mlflow.autolog()
```

In [ ]:
# Enable MLflow autologging for sklearn within this notebook

mlflow.autolog()

2026/05/03 23:30:16 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/05/03 23:30:16 WARNING mlflow.utils.autologging_utils: MLflow spark autologging is known to be compatible with 3.3.0 <= pyspark, but the installed version is 4.1.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/05/03 23:30:16 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark: MLflow Spark dataset autologging is not supported on Databricks shared clusters or Databricks serverless clusters.
2026/05/03 23:30:16 WARNING mlflow.utils.autologging_utils: MLflow pyspark.ml autologging is known to be compatible with 3.3.0 <= pyspark, but the installed version is 4.1.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/05/03 23:30:16 WARNING mlflow.tracking.fluent: Exception raised while e

Next, train a classifier within the context of an MLflow run, which automatically logs the trained model and many associated metrics and parameters. 

### 4. In the follwoing, start a run and give it a name `gb_baseline`:
```
with mlflow.start_run(run_name=[name]) as run:
```

The model's AUC score inot automatically logged, you need to manually log it using `mlflow.log_metric([metric_name],value)`

In [ ]:
# Start a new run and assign a run_name **gb_baseline** for future reference
with mlflow.start_run(run_name = 'gb_baseline') as run:

  model = sklearn.ensemble.GradientBoostingClassifier(random_state=0)

  # Models, parameters, and training metrics are tracked automatically
  model.fit(X_train, y_train)

  predicted_probs = model.predict_proba(X_test)
  roc_auc = sklearn.metrics.roc_auc_score(y_test, predicted_probs[:,1])

  print("Test AUC of: {}".format(roc_auc))

  # The AUC score on test data is not automatically logged, so log it manually
  mlflow.log_metric("test_auc", roc_auc)

🔗 View Logged Model at: https://dbc-c18777fb-f304.cloud.databricks.com/ml/experiments/4348360049629536/models/m-a12ac3927beb4c408e3f07e97755aa7e?o=7474660660881489

Test AUC of: 0.8834115358205539


### 5. Explore the Logged Model using the output URL 

- you may use the Experiment icon on the right tool bar to locate the experiment run. 

### 6. Adjust the Model parameters and log another run

Train another model with different hyperparameters.
- fit `model_2` with `n_estimators=200`.
- call the new run `gb_200_estimators`


In [ ]:
# Start a new run and assign a run_name for future reference
with mlflow.start_run(run_name = 'gb_200_estimators') as run:
  model_2 = sklearn.ensemble.GradientBoostingClassifier(
    random_state=0, 
    # Try a new parameter setting for n_estimators
    n_estimators=200,
  )
  model_2.fit(X_train, y_train)

  predicted_probs = model_2.predict_proba(X_test)
  roc_auc = sklearn.metrics.roc_auc_score(y_test, predicted_probs[:,1])

  print("Test AUC of: {}".format(roc_auc))
  
  # The AUC score on test data is not automatically logged, so log it manually
  mlflow.log_metric("test_auc", roc_auc)

🔗 View Logged Model at: https://dbc-c18777fb-f304.cloud.databricks.com/ml/experiments/4348360049629536/models/m-cac070d6151942a99b0843797ad05b55?o=7474660660881489

Test AUC of: 0.8912222476539253


### View MLflow runs
To view the logged training runs, click the **Experiment** icon at the upper right of the notebook to display the experiment sidebar. If necessary, click the refresh icon to fetch and monitor the latest runs. 

<img width="350" src="https://docs.databricks.com/_static/images/mlflow/quickstart/experiment-sidebar-icons.png"/>

You can then click the experiment page icon to display the more detailed MLflow experiment page ([AWS](https://docs.databricks.com/applications/mlflow/tracking.html#notebook-experiments)|[Azure](https://docs.microsoft.com/azure/databricks/applications/mlflow/tracking#notebook-experiments)|[GCP](https://docs.gcp.databricks.com/applications/mlflow/tracking.html#notebook-experiments)). This page allows you to compare runs and view details for specific runs.

<img width="800" src="https://docs.databricks.com/_static/images/mlflow/quickstart/compare-runs.png"/>

## Load Tracked Models
You can also access the results for a specific run using the MLflow API. The code in the following cell illustrates how to load the model trained in a given MLflow run and use it to make predictions. You can also find code snippets for loading specific models on the MLflow run page ([AWS](https://docs.databricks.com/applications/mlflow/tracking.html#view-notebook-experiment)|[Azure](https://docs.microsoft.com/azure/databricks/applications/mlflow/tracking#view-notebook-experiment)|[GCP](https://docs.gcp.databricks.com/applications/mlflow/tracking.html#view-notebook-experiment)).

To load the model as a regular Python function, you may use:

```python
func = mlflow.pyfunc.load_model('runs:/{run_id}/model')
```

To obtain `run_id` from `run_name`, you may use:

```python
mlflow.search_runs(filter_string="run_name='[run_name]'").run_id[0]
```


### 7. In the following, use saved `gb_200_estimators` model to make a prediction for `X_test` dataset

- and verify that we get the same results as using `model_2`. 

In [ ]:
# search for the model run_id
run_id = mlflow.search_runs(filter_string = "run_name = 'gb_200_estimators'").iloc[0].run_id

# After a model has been logged, you can load it in different notebooks or jobs
model_loaded = mlflow.pyfunc.load_model('runs:/{}/model'.format(run_id))

predictions_loaded = model_loaded.predict(X_test)
predictions_original = model_2.predict(X_test)

# The loaded model should match the original
assert(np.array_equal(predictions_loaded, predictions_original))

### 8. Search runs to retrieve the best model

Because all of the runs are tracked by MLflow, you can retrieve the metrics and parameters for the best run using the MLflow search runs API to find the tuning run with the highest test auc, order by `test_auc` (desecnding) and by `start_time` (latest first).

e.g.

```
search_runs(
  order_by=['metrics.[metric_name] DESC', '[variable]', 'params.[para_name]],
  max_results=
)
```

Also print the best run's
- ID: `bestrun.run_id`
- Name: `['tags.mlflow.runName']`
- Number of estimators: `['params.n_estimators']`
- Max Depth: `['params.max_depth']`
- Learning Rate: `['params.learning_rate']`

In [ ]:
# Sort runs by their test auc; in case of ties, use the most recent run
best_run = mlflow.search_runs(
  order_by = ['metrics.test_auc DESC', 'start_time DESC'],
  max_results = 10,
).iloc[0]


print('Best Run')
print(f"Run ID: {best_run.run_id}")
print(f"Run name:{best_run['tags.mlflow.runName']}")
print(f'AUC: {best_run["metrics.test_auc"]}')
print(f'Num Estimators: {best_run["params.max_depth"]}')
print(f'Max Depth: {best_run["params.max_depth"]}')
print(f'Learning Rate: {best_run["params.learning_rate"]}')

Best Run
Run ID: d6fa1708dbe444e9ad3f318841eac43b
Run name:gb_200_estimators
AUC: 0.8912222476539253
Num Estimators: 3
Max Depth: 3
Learning Rate: 0.1


### 9. Load the best model (using its run_id) to make some predictions

- load best model as `best_model_pyfunc`
- use the model to predict on the test data (first 5 rows)

In [ ]:
best_model_pyfunc = mlflow.pyfunc.load_model('runs:/{}/model'.format(best_run.run_id))
best_model_predictions = best_model_pyfunc.predict(X_test[:5])
print("Test Predictions: {}".format(best_model_predictions))

Test Predictions: [False False False False False]


Compare the predictions with the true labels

In [ ]:
print("Test true labels: {}".format(list(y_test[:5])))

Test true labels: [False, True, False, False, True]


### 10. Compare multiple runs in the UI
As in Part 1, you can view and compare the runs in the MLflow experiment details page, accessible via the external link icon at the top of the **Experiment** sidebar. 

On the experiment details page, click the "+" icon to expand the parent run, then select all runs except the parent, and click **Compare**. You can visualize the different runs using a parallel coordinates plot, which shows the impact of different parameter values on a metric. 

<img width="800" src="https://docs.databricks.com/aws/en/assets/images/parallel-coord-plot-f39e2027c85d0bfbce0c6c44b79d9e49.png"/>Parallel coordinates chart</img>

This is the end of the lab.